In [1]:
import os
import re
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.metrics import classification_report, confusion_matrix

# Включаем автоматическую перезагрузку связанных .py файлов без перезапуска ядра
%load_ext autoreload
%autoreload 2

# Отключаем визуальный спам предупреждений scikit-learn для чистоты научного отчета
warnings.filterwarnings("ignore")

# Подключаем наши спроектированные модули
import src.text_cleaner as tc
import src.ml_classifier as ml_clf

# Настройка визуализации графиков
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["font.size"] = 12

DATA_DIR = "data"
MODELS_DIR = "models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Входные и выходные файлы проекта
RAW_DATA_PATH = os.path.join(DATA_DIR, "data_merged.csv")
LABELED_DATA_PATH = os.path.join(DATA_DIR, "manual_labeling_sample.csv")
STAGE1_OUTPUT_PATH = os.path.join(DATA_DIR, "posts_preprocessed_stage1.csv")

In [2]:
if not os.path.exists(LABELED_DATA_PATH):
    raise FileNotFoundError(f"Файл разметки {LABELED_DATA_PATH} не найден!")

# 1. Загрузка размеченного вручную датасета
df_labeled_raw = pd.read_csv(LABELED_DATA_PATH)

# 2. Создание структуры согласно твоим требованиям
df_labeled = pd.DataFrame()
df_labeled["id"] = df_labeled_raw.index  # Используем индекс в качестве временного ID
df_labeled["original_text"] = df_labeled_raw["text"].fillna("").astype(str)

# 3. Применение глубокой очистки (генерация технического поля 'text' с точками)
print("Начало глубокой очистки текстов. Удаление стоп-слов, ссылок и мусора...")
df_labeled["text"] = df_labeled["original_text"].apply(tc.clean_raw_text)

# Сохраняем метку разметки для обучения
df_labeled["preliminary_label"] = df_labeled_raw["preliminary_label"]

print(f"\nПредобработка завершена. Успешно очищено {len(df_labeled)} записей.")

Начало глубокой очистки текстов. Удаление стоп-слов, ссылок и мусора...

Предобработка завершена. Успешно очищено 1200 записей.


In [3]:
# ==============================================================================
# ЯЧЕЙКА 4: ШАГ 1 — БАЗОВАЯ МОДЕЛЬ (БЕЗ ШТРАФОВ)
# ==============================================================================
# Отсекаем мусор, оставляя чистые вакансии (+1) и резюме (-1)
df_train = df_labeled[df_labeled["preliminary_label"].isin(["vacancy", "resume"])].copy()
X_train = df_train["text"].values
y_train = np.where(df_train["preliminary_label"] == "vacancy", 1, -1)

# Создаем одиночную модель без штрафов
baseline_pipeline = ml_clf.create_single_pipeline(penalty="none", C=1.0)
baseline_pipeline.fit(X_train, y_train)

# Кросс-валидация базовой модели
y_pred_oof_base = ml_clf.get_binary_oof_predictions(X_train, y_train, baseline_pipeline)

print("=== ШАГ 1: МЕТРИКИ БАЗОВОЙ МОДЕЛИ (БЕЗ ШТРАФОВ) ===")
print(classification_report(y_train, y_pred_oof_base, labels=[-1, 1], target_names=["Resume (-1)", "Vacancy (+1)"], zero_division=0))

# Топ-словосочетания без регуляризации с выводом весов
base_features = ml_clf.extract_binary_feature_importance(baseline_pipeline, top_n=10)
print("\n=== ВЕСА СЛОВ БАЗОВОЙ МОДЕЛИ (МНОГО ШУМА) ===")
print("Индикаторы ВАКАНСИИ (+1):")
for rank, (word, weight) in enumerate(base_features["Vacancy"], 1):
    print(f"  {rank}. {word:<25} (вес: +{weight})")

print("\nИндикаторы РЕЗЮМЕ (-1):")
for rank, (word, weight) in enumerate(base_features["Resume"], 1):
    print(f"  {rank}. {word:<25} (вес: {weight})")

=== ШАГ 1: МЕТРИКИ БАЗОВОЙ МОДЕЛИ (БЕЗ ШТРАФОВ) ===
              precision    recall  f1-score   support

 Resume (-1)       0.99      0.99      0.99       164
Vacancy (+1)       1.00      1.00      1.00       865

    accuracy                           1.00      1029
   macro avg       1.00      1.00      1.00      1029
weighted avg       1.00      1.00      1.00      1029


=== ВЕСА СЛОВ БАЗОВОЙ МОДЕЛИ (МНОГО ШУМА) ===
Индикаторы ВАКАНСИИ (+1):
  1. подробности               (вес: +8.9831)
  2. вакансия                  (вес: +8.7655)
  3. сайте                     (вес: +8.3696)
  4. подробности контакты      (вес: +8.2541)
  5. детали                    (вес: +7.3351)
  6. условия                   (вес: +7.2878)
  7. требуется                 (вес: +7.1999)
  8. ищем                      (вес: +7.0693)
  9. работа                    (вес: +6.7715)
  10. детали сайте              (вес: +6.7321)

Индикаторы РЕЗЮМЕ (-1):
  1. помогу                    (вес: -15.7393)
  2. предлагаю 

In [ ]:
# ==============================================================================
# ЯЧЕЙКА 5: ШАГ 2 — МОДЕЛЬ С L1-РЕГУЛЯРИЗАЦИЕЙ (ОТБОР ПРИЗНАКОВ / LASSO)
# ==============================================================================
l1_pipeline = ml_clf.create_single_pipeline(penalty="l1", C=1.0)
l1_pipeline.fit(X_train, y_train)

# Кросс-валидация L1 модели
y_pred_oof_l1 = ml_clf.get_binary_oof_predictions(X_train, y_train, l1_pipeline)

print("=== ШАГ 2: МЕТРИКИ МОДЕЛИ С L1-РЕГУЛЯРИЗАЦИЕЙ ===")
print(classification_report(y_train, y_pred_oof_l1, labels=[-1, 1], target_names=["Resume (-1)", "Vacancy (+1)"], zero_division=0))

# Топ-словосочетания L1 с выводом весов
l1_features = ml_clf.extract_binary_feature_importance(l1_pipeline, top_n=10)
print("\n=== ВЕСА СЛОВ L1-МОДЕЛИ ===")
print("Индикаторы ВАКАНСИИ (+1):")
if l1_features["Vacancy"]:
    for rank, (word, weight) in enumerate(l1_features["Vacancy"], 1):
        print(f"  {rank}. {word:<25} (вес: +{weight})")
else:
    print("Все веса вакансий обнулены")

print("\nИндикаторы РЕЗЮМЕ (-1):")
for rank, (word, weight) in enumerate(l1_features["Resume"], 1):
    print(f"  {rank}. {word:<25} (вес: {weight})")

# Подсчитываем зануление признаков
l1_coefs = l1_pipeline.named_steps["clf"].coef_[0]
zeroed_features = np.sum(l1_coefs == 0)
print(f"\nНаучный факт: L1-регуляризация занулила {zeroed_features} нерелевантных признаков из {len(l1_coefs)}!")

=== ШАГ 2: МЕТРИКИ МОДЕЛИ С L1-РЕГУЛЯРИЗАЦИЕЙ ===
              precision    recall  f1-score   support

 Resume (-1)       0.99      0.96      0.98       164
Vacancy (+1)       0.99      1.00      1.00       865

    accuracy                           0.99      1029
   macro avg       0.99      0.98      0.99      1029
weighted avg       0.99      0.99      0.99      1029


=== ВЕСА СЛОВ L1-МОДЕЛИ ===
Индикаторы ВАКАНСИИ (+1):
  [ВНИМАНИЕ] Нет активных положительных признаков! Все веса вакансий обнулены L1 штрафом.

Индикаторы РЕЗЮМЕ (-1):
  1. помогу                    (вес: -59.8802)
  2. услуги                    (вес: -22.6109)
  3. кейсы                     (вес: -15.0738)
  4. достижения                (вес: -13.8641)
  5. резюме                    (вес: -10.2046)
  6. профиля                   (вес: -1.4602)
  7. предлагаю                 (вес: -0.1019)

Научный факт: L1-регуляризация занулила 4993 нерелевантных признаков из 5000!


In [5]:
# ==============================================================================
# ЯЧЕЙКА 6: ШАГ 3 — МОДЕЛЬ С L2-РЕГУЛЯРИЗАЦИЕЙ (СЖАТИЕ ВЕСОВ / RIDGE)
# ==============================================================================
l2_pipeline = ml_clf.create_single_pipeline(penalty="l2", C=1.0)
l2_pipeline.fit(X_train, y_train)

# Кросс-валидация L2 модели
y_pred_oof_l2 = ml_clf.get_binary_oof_predictions(X_train, y_train, l2_pipeline)

print("=== ШАГ 3: МЕТРИКИ МОДЕЛИ С L2-РЕГУЛЯРИЗАЦИЕЙ ===")
print(classification_report(y_train, y_pred_oof_l2, labels=[-1, 1], target_names=["Resume (-1)", "Vacancy (+1)"], zero_division=0))

# Топ-словосочетания L2 с выводом весов
l2_features = ml_clf.extract_binary_feature_importance(l2_pipeline, top_n=10)
print("\n=== ВЕСА СЛОВ L2-МОДЕЛИ (ВЕСА СЖАТЫ, НО НЕ ОБНУЛЕНЫ) ===")
print("Индикаторы ВАКАНСИИ (+1):")
for rank, (word, weight) in enumerate(l2_features["Vacancy"], 1):
    print(f"  {rank}. {word:<25} (вес: +{weight})")

print("\nИндикаторы РЕЗЮМЕ (-1):")
for rank, (word, weight) in enumerate(l2_features["Resume"], 1):
    print(f"  {rank}. {word:<25} (вес: {weight})")

=== ШАГ 3: МЕТРИКИ МОДЕЛИ С L2-РЕГУЛЯРИЗАЦИЕЙ ===
              precision    recall  f1-score   support

 Resume (-1)       0.99      0.98      0.99       164
Vacancy (+1)       1.00      1.00      1.00       865

    accuracy                           1.00      1029
   macro avg       1.00      0.99      0.99      1029
weighted avg       1.00      1.00      1.00      1029


=== ВЕСА СЛОВ L2-МОДЕЛИ (ВЕСА СЖАТЫ, НО НЕ ОБНУЛЕНЫ) ===
Индикаторы ВАКАНСИИ (+1):
  1. подробности               (вес: +1.0585)
  2. условия                   (вес: +0.9953)
  3. сайте                     (вес: +0.9827)
  4. подробности контакты      (вес: +0.9768)
  5. требования                (вес: +0.9564)
  6. умение                    (вес: +0.9475)
  7. работа                    (вес: +0.9392)
  8. вакансия                  (вес: +0.8466)
  9. обязанности               (вес: +0.8266)
  10. ищем                      (вес: +0.8111)

Индикаторы РЕЗЮМЕ (-1):
  1. помогу                    (вес: -3.5138)
  2. ус

In [ ]:
# ==============================================================================
# ЯЧЕЙКА 7: ШАГ 4 — АВТОМАТИЧЕСКАЯ ОПТИМИЗАЦИЯ СЕТКИ ПАРАМЕТРОВ L2
# ==============================================================================
# Строим сетку поиска строго на базе L2-регуляризации, доказав её научное превосходство
grid_search, base_pipeline = ml_clf.build_binary_pipeline()

print("Запуск финального GridSearchCV поиска лучших параметров...")
grid_search.fit(X_train, y_train)

print("\n===========================================")
print("=== РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ ДЛЯ СЛАЙДОВ ===")
print("===========================================")
print(f"Оптимальный тип регуляризации: {grid_search.best_params_['clf__penalty'].upper()}")
print(f"Оптимальная сила штрафа C     : {grid_search.best_params_['clf__C']}")
print(f"Наилучший Macro F1-score      : {grid_search.best_score_:.4f}")
print("===========================================")

# Извлекаем лучшую модель
best_model = grid_search.best_estimator_

# Сохранение обученной модели
joblib.dump(best_model, os.path.join(MODELS_DIR, "best_binary_classifier.joblib"))
print("\nОптимальная модель успешно экспортирована на диск.")

Запуск финального GridSearchCV поиска лучших параметров L2...

=== РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ ДЛЯ СЛАЙДОВ ===
Оптимальный тип регуляризации: L2
Оптимальная сила штрафа C     : 5.0
Наилучший Macro F1-score      : 0.9945

Оптимальная модель успешно экспортирована на диск.


In [7]:
# ==============================================================================
# ЯЧЕЙКА 8: ЭКСПОРТ РЕЗУЛЬТАТОВ ЭТАПА I В ШКАЛЕ [-1.0, +1.0]
# ==============================================================================
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(f"Файл сырых данных {RAW_DATA_PATH} не найден!")

# 1. Загрузка сырых постов из Telegram
df_raw = pd.read_csv(RAW_DATA_PATH)

# Создаем чистую структуру под твои требования
df_final = pd.DataFrame()
df_final["id"] = df_raw["message_id"].fillna(0).astype(int)
df_final["original_text"] = df_raw["text"].fillna("").astype(str)

# 2. Глубокая очистка технического поля text (точки сохранены)
print("Применение лингвистической очистки к сырому датасету...")
df_final["text"] = df_final["original_text"].apply(tc.clean_raw_text)

# 3. Предсказание вероятностей лучшей моделью
print("Классификация постов на основе сохраненного лучшего классификатора...")
probs_raw = best_model.predict_proba(df_final["text"].values)[:, 1] # Вероятность класса 1 (vacancy)

# Математическая трансформация в шкалу [-1, 1]
df_final["p_vacancy_resume_between"] = ml_clf.transform_prob_to_signed_score(probs_raw)

# 4. Присвоение категорий уверенности на основе градации
df_final["confidence_tier"] = df_final["p_vacancy_resume_between"].apply(ml_clf.get_confidence_tier)

print("\nРаспределение постов по полученным классам уверенности:")
print(df_final["confidence_tier"].value_counts())

# Экспортируем ровно 5 требуемых полей
final_columns = ["id", "original_text", "text", "p_vacancy_resume_between", "confidence_tier"]
df_export = df_final[final_columns].copy()
df_export.to_csv(STAGE1_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"\nЭтап I успешно завершен. Результаты сохранены в: {STAGE1_OUTPUT_PATH}")
display(df_export.head(5))

Применение лингвистической очистки к сырому датасету...
Классификация постов на основе сохраненного лучшего классификатора...

Распределение постов по полученным классам уверенности:
confidence_tier
Ultra-Strict Vacancy    19593
Strict Vacancy           6814
Resume                   1841
Moderate Vacancy         1691
Trash / Uncertain         498
Mild Vacancy              271
Name: count, dtype: int64

Этап I успешно завершен. Результаты сохранены в: data\posts_preprocessed_stage1.csv


,id,original_text,text,p_vacancy_resume_between,confidence_tier
0,5916,Резюме: **Marketing manager**\n\n__👩‍💻 Карташе...,резюме marketing manager. карташева анна. ник ...,-0.340668,Trash / Uncertain
1,5915,**SMM-менеджер в творческую студию Звезд\n\nРа...,smm менеджер творческую студию звезд. работода...,0.884587,Strict Vacancy
2,5913,Резюме: **Бренд-менеджер / Проджект-менеджер**...,резюме бренд менеджер проджект менеджер. шабан...,-0.122221,Trash / Uncertain
3,5909,**Контент-менеджер на управление контент произ...,контент менеджер управление контент производст...,0.942931,Ultra-Strict Vacancy
4,5908,__Резюме:__ **Руководитель отдела PR и внешних...,резюме руководитель отдела pr внешних коммуник...,-0.356177,Trash / Uncertain
